In [ ]:
!pip install -q roboflow ultralytics

In [ ]:
import os
from pathlib import Path
from google.colab import drive, userdata

drive.mount('/content/drive')
roboflow_token = userdata.get('ROBOFLOW_API_KEY')

RESULTS_DIR = '/content/drive/MyDrive/ECS-174-Project/detector'
MODEL = 'yolov8m-seg.pt'   # pretrained COCO weights
BATCH = 3
os.makedirs(RESULTS_DIR, exist_ok=True)

In [ ]:
from roboflow import Roboflow
rf = Roboflow(api_key=roboflow_token)
project = rf.workspace("bouldering-hold-labeling").project("bouldering-holds-stage-1")
version = project.version(3) # resize off version + removed box imgs + removed empty img
dataset = version.download("yolov8")
print(dataset.location)

In [ ]:
import yaml

DATA_YAML = os.path.join(dataset.location, 'data.yaml')
with open(DATA_YAML) as f:
    data = yaml.safe_load(f)

data['path'] = dataset.location
data['train'] = 'train/images'
data['val'] = 'valid/images'
data['test'] = 'test/images'
with open(DATA_YAML, 'w') as f:
    yaml.safe_dump(data, f)

print('Classes:', data['names'])

In [ ]:
import glob

total = 0
for split in ['train', 'valid', 'test']:
    n = len(glob.glob(f'{dataset.location}/{split}/images/*'))
    total += n
    print(f'{split}: {n} images')
print('total:', total)

In [ ]:
from ultralytics import YOLO

def train_run(name, imgsz, epochs=100, patience=20):
    """Train a run, or resume it if a previous session was interrupted."""
    last = Path(RESULTS_DIR) / name / 'weights' / 'last.pt'
    if last.exists():
        print(f'Resuming {name} from {last}')
        try:
            YOLO(str(last)).train(resume=True)
        except AssertionError as e:      # raised when the run already finished
            print(f'{name} already finished: {e}')
    else:
        model = YOLO(MODEL)              # start from pretrained COCO weights
        model.train(
            data=DATA_YAML,
            imgsz=imgsz,
            epochs=epochs,
            patience=patience,           # early stopping
            batch=BATCH,                   
            cache='disk',                # resize each image once, reuse every epoch
            project=RESULTS_DIR,
            name=name,
            exist_ok=True,
            seed=0,                      # same seed across runs for a fair comparison
        )
    return Path(RESULTS_DIR) / name

In [ ]:
run_1280 = train_run('roundA_1280', imgsz=1280)

In [ ]:
run_1024 = train_run('roundA_1024', imgsz=1024)

In [ ]:
def seg_metrics(run_dir, imgsz, split):
    model = YOLO(str(Path(run_dir) / 'weights' / 'best.pt'))
    m = model.val(data=DATA_YAML, imgsz=imgsz, split=split, max_det=500, batch=BATCH * 2, verbose=False)
    rows = {'all': (m.seg.map50, m.seg.map)}
    for i, c in enumerate(m.ap_class_index):
        _, _, ap50, ap = m.seg.class_result(i)
        rows[model.names[int(c)]] = (ap50, ap)
    print(f'\n{Path(run_dir).name} ({split}):')
    for k, (ap50, ap) in rows.items():
        print(f'  {k:7s} mask mAP50 {ap50:.3f} | mAP50-95 {ap:.3f}')
    return rows

val_1280 = seg_metrics(RESULTS_DIR + '/roundA_1280', 1280, 'val')
val_1024 = seg_metrics(RESULTS_DIR + '/roundA_1024', 1024, 'val')

In [ ]:
CHOSEN, CHOSEN_IMGSZ = 'roundA_1280', 1280
test_scores = seg_metrics(f'{RESULTS_DIR}/{CHOSEN}', CHOSEN_IMGSZ, 'test')

In [ ]:
import glob
n = sum(l.split()[0] == '1' for f in glob.glob(f'{dataset.location}/test/labels/*.txt') for l in open(f) if l.strip())
print('volumes in test:', n)

In [ ]:
import pandas as pd

LOG = f'{RESULTS_DIR}/results_log.csv'
rows = []
for name, imgsz, scores, split in [('roundA_1280', 1280, val_1280, 'val'),
                                   ('roundA_1024', 1024, val_1024, 'val'),
                                   (CHOSEN, CHOSEN_IMGSZ, test_scores, 'test')]:
    for cls, (ap50, ap) in scores.items():
        rows.append({'run': name, 'model': MODEL, 'imgsz': imgsz, 'split': split,
                     'class': cls, 'mask_mAP50': round(ap50, 4), 'mask_mAP50_95': round(ap, 4)})
log = pd.DataFrame(rows)
log.to_csv(LOG, mode='a', header=not os.path.exists(LOG), index=False)
display(log)